# CIFAR-10 Classification

## 1.3 New Model Architecture + Data Augmentation

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import time
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import shutil

In [2]:
import numpy as np
import random
import os

seed = 42
# Setting Python and NumPy seeds
random.seed(seed)
np.random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
    
# Setting PyTorch seeds (CPU and all GPUs)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
    
# Ensure deterministic behavior in cuDNN
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
# checking if CUDA is available and set the device accordingly,
# if a GPU is available it will use it for computations, otherwise it will fall back to using the CPU.
# in this case I am using RTX 3060 GPU which has CUDA support
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Assuming that we are on a CUDA machine, this should print a CUDA device:
print(device)

cuda:0


## PreProcessing


In [ ]:
train_transform = transforms.Compose(
    [
        # adding random cropping by increasing the padding to 4 so the size
        # of the image increases from 32x32 to 40x40 then randomly cropping
        # the image to 32x32 again
        transforms.RandomCrop(32, padding=4),
        # horizontal flipping each image by 50% probability meaning each image has
        # 50% probability of being flipped horizontaly
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        # using the dataset mean and std instead of using 0.5 to keep it in range of 0-1
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                (0.2023, 0.1994, 0.2010))
    ]
)

test_transform = transforms.Compose(
    [
        # not applying augmentation to test dataset because we want the test
        # data not to be noisy and unfair we only want to introduce this to
        # the training dataset
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
            (0.2023, 0.1994, 0.2010))
    ]
)

batch_size = 64

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=False, transform=train_transform)
trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                        download=False, transform=test_transform)
testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

## Resnet 18 Architecture

![Cifar Res Net 18](resnet18_full_visual_flow_readable.png)